# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This audit validates Week 5 work for Lane 2 — Refresh / Content Opportunity Scoring. It examines two research-paper findings, reproduces an honest client-grouped Week 5-style evaluation, compares it with a weaker random-row diagnostic, audits leakage, reviews real failures, and rewrites claims to match the evidence.

## 1. Two research-paper findings + my methodology questions

**Finding 1: The paper reports 71% logistic-regression holdout accuracy for growth-versus-decline classification.** The extracted Growth & Classification appendix says logistic regression achieved 71% holdout accuracy while describing which sampled features separate growing pages from declining pages. This was placed in an exploratory ML appendix and is not presented as headline portfolio evidence.

Constructive methodology question 1: How was growth versus decline defined, which exact feature window and label window were used, and was the 71% result calculated on held-out rows or an already optimized sample?

Why it matters: A label definition and outcome window determine what the accuracy means. If the label reused contemporary or future-derived measurements as inputs, or if training and testing were selected without window separation, the reported figure would be difficult to interpret as generalization.

**Finding 2: The direct portfolio study reports 341,701 content pieces across 57 brands.** The paper uses direct aggregate comparisons as its main evidence base, then treats its ML appendix as exploratory.

Constructive methodology question 2: How were pages and brands selected for the direct portfolio comparison, and could the same clients or related page observations affect comparisons that are presented as independent evidence?

Why it matters: Sample coverage, panel selection, and dependencies among pages from the same client affect how broadly a portfolio-level finding transfers. The paper itself says direct aggregate comparisons lead the study, while content-age confounding and observational interpretation remain limitations.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download
from pypdf import PdfReader


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root not found')

repo_root = find_repo_root(Path.cwd().resolve())
token = os.getenv('HF_TOKEN')
env_path = next((base / '.env' for base in [repo_root, *repo_root.parents] if (base / '.env').exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
print('Local Hugging Face authentication configured:', bool(token))

paper_path = repo_root / 'docs' / 'flyrank-seo-research-march-2026.pdf'
reader = PdfReader(str(paper_path))
paper_text = chr(10).join((page.extract_text() or '') for page in reader.pages)
print('Paper pages:', len(reader.pages))
print('Paper text characters:', len(paper_text))
print('Selected finding supported by extracted text:', '71%' in paper_text and '341,701' in paper_text)
assert len(reader.pages) > 0
assert '71%' in paper_text
assert '341,701' in paper_text


Local Hugging Face authentication configured: True


Paper pages: 36
Paper text characters: 53385
Selected finding supported by extracted text: True


## 2. My model under an honest split (before/after)

I reproduce Week 5 with the same five March 1–15 features and `is_declining_proxy` target. The honest **reference** remains a client-grouped holdout: `GroupShuffleSplit` by `client_hash_id` with a fixed seed of 42 and approximately 20% of clients reserved for testing.

The evaluation population is corrected so that candidate membership is based only on March 1–15 information. March 16–31 availability and the proxy label are joined separately. A page needs 16 available late-period dates to receive an evaluable label. The code reports both the early-only candidate count and the evaluable-later-label subset; all reported metrics below apply only to that evaluable subset.

The additional **random-row split** remains a deliberately weaker diagnostic. It may place pages from the same client in both training and testing, which can overstate generalization by letting the model reuse client-specific patterns.

All splits use the same model settings: `StandardScaler`, `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`. Rankings use predicted probabilities. Precision@20 and Precision@50 use the positive proxy label among the top K ranked pages. The base rate is the positive-class proportion of the relevant evaluable test set.

A time-aware evaluation is not substituted here because the available request is to use the March mid-panel development data while preserving the Week 5 grouped design.

In [2]:
import duckdb
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
    token=token,
)

con = duckdb.connect()
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_avg_position * gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15'
                             AND gsc_data_available IS TRUE
                        THEN gsc_impressions ELSE 0 END), 0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15'
                            AND gsc_data_available IS TRUE
                        THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_impressions ELSE 0 END) AS late_impressions,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15'
                            AND gsc_data_available IS TRUE
                        THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  GROUP BY client_hash_id, content_hash_id
), early_candidates AS (
  SELECT *,
         CASE WHEN early_impressions > 0
              THEN 100.0 * early_clicks / early_impressions ELSE NULL END AS early_ctr
  FROM page_windows
  WHERE early_impressions > 0
), evaluable AS (
  SELECT *,
         CASE WHEN late_available_days = 16
                   AND late_impressions / 16.0 < early_impressions / 15.0
              THEN 1 ELSE 0 END AS is_declining_proxy
  FROM early_candidates
  WHERE late_available_days = 16
    AND weighted_avg_position IS NOT NULL
    AND early_ctr IS NOT NULL
)
SELECT client_hash_id, content_hash_id, early_impressions, early_clicks,
       weighted_avg_position, active_search_days, early_ctr, is_declining_proxy
FROM evaluable
'''
df = con.execute(feature_query, [march_url]).fetchdf()

# Candidate and evaluability counts are kept in separate SQL-derived stages.
population_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS early_impressions,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS early_available_days,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  GROUP BY client_hash_id, content_hash_id
)
SELECT
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0) AS early_candidate_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days = 16) AS evaluable_later_label_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days >= 1 AND late_available_days < 16) AS partial_late_availability_pages
FROM page_windows
'''
pop_counts = con.execute(population_query, [march_url]).fetchone()
early_candidate_rows, evaluable_later_rows, partial_late_rows = pop_counts
feature_columns = ['early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']
X = df[feature_columns].copy()
y = df['is_declining_proxy'].astype(int)
print('Development month: 2026-03')
print('Early-period candidate pages (early availability only):', early_candidate_rows)
print('Pages with evaluable late label (16 available late dates):', evaluable_later_rows)
print('Pages with partial late availability (1-15 dates):', partial_late_rows)
print('Measured evaluation pages:', len(df))
print('Unique client/page pairs:', df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0])
assert len(df) == df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0]
assert len(df) <= evaluable_later_rows

# Client-grouped split, matching the Week 5 GroupShuffleSplit logic.
groups = df['client_hash_id'].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
g_train, g_test = next(gss.split(X, y, groups=groups))

def precision_at_k(frame, probability_column, k):
    top = frame.sort_values(probability_column, ascending=False, kind='mergesort').head(k)
    return float(top['is_declining_proxy'].mean()) if len(top) else 0.0

def fit_and_evaluate(train_pos, test_pos):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ])
    model.fit(X.iloc[train_pos], y.iloc[train_pos])
    result = df.iloc[test_pos].copy()
    result['model_probability'] = model.predict_proba(X.iloc[test_pos])[:, 1]
    return model, result

model_grouped, grouped_test = fit_and_evaluate(g_train, g_test)
train_df = df.iloc[g_train]
train_clients = set(train_df['client_hash_id'].astype(str))
test_clients = set(grouped_test['client_hash_id'].astype(str))

r_train, r_test = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42, stratify=y)
model_random, random_test = fit_and_evaluate(r_train, r_test)
random_train_clients = set(df.iloc[r_train]['client_hash_id'].astype(str))
random_test_clients = set(random_test['client_hash_id'].astype(str))

comparison = pd.DataFrame([
    {
        'Split':'Client-grouped (honest reference)','Train pages':len(g_train),'Test pages':len(g_test),
        'Train clients':len(train_clients),'Test clients':len(test_clients),'Client overlap':len(train_clients.intersection(test_clients)),
        'Precision@20':precision_at_k(grouped_test,'model_probability',20),'Precision@50':precision_at_k(grouped_test,'model_probability',50),
        'ROC-AUC':roc_auc_score(grouped_test['is_declining_proxy'], grouped_test['model_probability']),
        'Positive-class/base rate':float(grouped_test['is_declining_proxy'].mean()),
    },
    {
        'Split':'Random row split (diagnostic only)','Train pages':len(r_train),'Test pages':len(r_test),
        'Train clients':len(random_train_clients),'Test clients':len(random_test_clients),'Client overlap':len(random_train_clients.intersection(random_test_clients)),
        'Precision@20':precision_at_k(random_test,'model_probability',20),'Precision@50':precision_at_k(random_test,'model_probability',50),
        'ROC-AUC':roc_auc_score(random_test['is_declining_proxy'], random_test['model_probability']),
        'Positive-class/base rate':float(random_test['is_declining_proxy'].mean()),
    },
])
print(comparison.to_string(index=False, float_format=lambda x:f'{x:.4f}'))
print('Metrics apply only to the evaluable 16-date late-period subset:', len(df))
assert len(train_clients.intersection(test_clients)) == 0
assert grouped_test['is_declining_proxy'].nunique() == 2
assert random_test['is_declining_proxy'].nunique() == 2


Development month: 2026-03
Early-period candidate pages (early availability only): 151981
Pages with evaluable late label (16 available late dates): 85565
Pages with partial late availability (1-15 dates): 55902
Measured evaluation pages: 85565
Unique client/page pairs: 85565
                             Split  Train pages  Test pages  Train clients  Test clients  Client overlap  Precision@20  Precision@50  ROC-AUC  Positive-class/base rate
 Client-grouped (honest reference)        81452        4113             31             8               0        0.5500        0.4800   0.7667                    0.3221
Random row split (diagnostic only)        68452       17113             38            34              33        0.8500        0.8000   0.7056                    0.4183
Metrics apply only to the evaluable 16-date late-period subset: 85565


The before/after comparison is now based on actual held-out predictions. The grouped split is the honest reference. A random split is weaker because same-client observations are split across partitions; if its metrics are higher, that does not prove deployment skill. The random split is useful only as evidence that validation design changes the result.

The grouped holdout uses only 9 test clients. That small client count limits confidence even when the page count is larger, because one unusual client can strongly affect the test result.

## 3. Leakage audit

**Audit conclusion:** The five final predictive features are safe under the March 1–15 timeline and do not contain label-derived values. Candidate population construction uses only March 1–15 availability and impressions. March 16–31 data is used only afterward to determine whether a full 16-date label is evaluable and to calculate the proxy.

This corrected population excludes partial late-period histories: a page must have positive early impressions and 16 available late-period dates to enter the measured evaluation frame. Pages with 1–15 late-period available dates are reported separately and are not scored. This is a label-availability criterion, not a predictive feature. It may still create survivorship or availability selection, which is documented as a limitation.

The audit below separates retained features, target-derived fields, future fields, identifiers, and product/action fields.

In [3]:
leakage_table = pd.DataFrame([
    {'Field':'early_impressions','What it measures':'GSC impressions summed March 1-15','Dates':'2026-03-01 to 2026-03-15','At prediction time?':'Yes','Future/target derived?':'No','Decision':'RETAIN'},
    {'Field':'early_clicks','What it measures':'GSC clicks summed March 1-15','Dates':'2026-03-01 to 2026-03-15','At prediction time?':'Yes','Future/target derived?':'No','Decision':'RETAIN'},
    {'Field':'weighted_avg_position','What it measures':'Impression-weighted GSC position in March 1-15','Dates':'2026-03-01 to 2026-03-15','At prediction time?':'Yes','Future/target derived?':'No','Decision':'RETAIN'},
    {'Field':'active_search_days','What it measures':'Distinct available GSC dates in March 1-15','Dates':'2026-03-01 to 2026-03-15','At prediction time?':'Yes','Future/target derived?':'No','Decision':'RETAIN'},
    {'Field':'early_ctr','What it measures':'100 x early clicks / early impressions','Dates':'2026-03-01 to 2026-03-15','At prediction time?':'Yes','Future/target derived?':'No, but derived from retained fields','Decision':'RETAIN'},
    {'Field':'is_declining_proxy','What it measures':'Late daily impressions below early daily impressions','Dates':'2026-03-16 to 2026-03-31','At prediction time?':'No','Future/target derived?':'Yes: target','Decision':'EXCLUDE FROM FEATURES'},
    {'Field':'late_impressions / late_clicks','What it measures':'March 16-31 traffic totals','Dates':'2026-03-16 to 2026-03-31','At prediction time?':'No','Future/target derived?':'Future window and close to target signal','Decision':'EXCLUDE'},
    {'Field':'trend_pct / trend_direction','What it measures':'Starter-dataset trend fields derived from period movement','Dates':'Starter trailing 90-day/sibling derivation','At prediction time?':'No for this warehouse target frame','Future/target derived?':'Target-derived sibling fields','Decision':'EXCLUDE'},
    {'Field':'client_hash_id / content_hash_id','What it measures':'Pseudonymized grouping identifiers','Dates':'Not a measurement window','At prediction time?':'Available as context','Future/target derived?':'No, but ID memorization risk','Decision':'GROUP/SPLIT/IDENTIFY ONLY; NEVER FEATURE'},
    {'Field':'product flags / action labels','What it measures':'Existing workflow decisions or constructed scores','Dates':'May follow decision or label logic','At prediction time?':'Circular risk','Future/target derived?':'Potentially encodes the answer','Decision':'EXCLUDE FROM MODEL; BASELINE ONLY'},
])
print(leakage_table.to_string(index=False))
assert set(feature_columns) == {'early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr'}
assert 'is_declining_proxy' not in feature_columns
assert 'late_impressions' not in feature_columns
assert 'late_clicks' not in feature_columns
print('Predictive feature count:', len(feature_columns))
print('Feature-window leakage check passed:', True)


                           Field                                          What it measures                                      Dates                At prediction time?                   Future/target derived?                                 Decision
               early_impressions                         GSC impressions summed March 1-15                   2026-03-01 to 2026-03-15                                Yes                                       No                                   RETAIN
                    early_clicks                              GSC clicks summed March 1-15                   2026-03-01 to 2026-03-15                                Yes                                       No                                   RETAIN
           weighted_avg_position            Impression-weighted GSC position in March 1-15                   2026-03-01 to 2026-03-15                                Yes                                       No                                   RET

### What the five features measure and why they are safe

- `early_impressions` measures accumulated demand from March 1–15 and is available before the March 16–31 proxy window.
- `early_clicks` measures accumulated clicks in the same early window and is not calculated from the late outcome.
- `weighted_avg_position` uses early impressions as weights and early dates as the measurement source; it has no late-window component.
- `active_search_days` counts early available dates and documents whether a page was active during the feature window.
- `early_ctr` is early clicks divided by early impressions. It is a derived feature but uses only allowed early information, so it is not itself label-derived.

The following are unsafe: late impressions/clicks as inputs; `is_declining_proxy` as a feature; `trend_pct` and `trend_direction`; IDs as predictors; and product flags or action labels.

## 4. Real failure examples

The following examples come from actual grouped-holdout predictions on the evaluable late-label subset. Anonymized IDs are shown only.

**False positives:** top-50 pages with proxy label 0, which are actual ranking false positives under the ranking definition of reviewing the top 50.

**Low-ranked positive examples:** proxy-positive pages outside the top 50. No classification threshold was defined or applied, so these are not conventional false negatives. Conventional false negatives would require an explicit probability threshold; it is not calculated here.

In [4]:
# Inspect actual top-50 false positives and low-ranked positive examples.
ranked_grouped = grouped_test.sort_values(['model_probability','client_hash_id','content_hash_id'], ascending=[False,True,True], kind='mergesort').copy()
ranked_grouped['rank'] = np.arange(1, len(ranked_grouped)+1)

false_positives = ranked_grouped[(ranked_grouped['rank'] <= 50) & (ranked_grouped['is_declining_proxy']==0)].copy()
low_ranked_positives = ranked_grouped[(ranked_grouped['rank'] > 50) & (ranked_grouped['is_declining_proxy']==1)].copy()

print('Grouped-holdout false positives in the top 50:', len(false_positives))
print('Positive-proxy pages outside the top 50 (low-ranked positive examples):', len(low_ranked_positives))
print('Classification threshold: none; rank-only review terminology used.')

columns = ['client_hash_id','content_hash_id','model_probability','is_declining_proxy','early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']
print('Actual false-positive examples in top 50 (maximum 2):')
print(false_positives[columns].head(2).to_string(index=False, float_format=lambda x:f'{x:.4f}'))
print('Actual low-ranked positive examples outside top 50 (maximum 2, lowest probability):')
print(low_ranked_positives.sort_values('model_probability', ascending=True)[columns].head(2).to_string(index=False, float_format=lambda x:f'{x:.4f}'))

if len(false_positives) == 0:
    print('No grouped-holdout false positives are available in the top 50.')
if len(low_ranked_positives) == 0:
    print('No positive-label pages are outside the top 50 in this held-out set.')

assert len(false_positives) >= 2
assert len(low_ranked_positives) >= 2


Grouped-holdout false positives in the top 50: 26
Positive-proxy pages outside the top 50 (low-ranked positive examples): 1301
Classification threshold: none; rank-only review terminology used.
Actual false-positive examples in top 50 (maximum 2):
         client_hash_id          content_hash_id  model_probability  is_declining_proxy  early_impressions  early_clicks  weighted_avg_position  active_search_days  early_ctr
client_a80fca3f171ed1de content_9540d884af3e41fd             0.9592                   0         31149.0000        7.0000                 6.6826                  15     0.0225
client_a80fca3f171ed1de content_f8758c939c0b9034             0.8019                   0         19309.0000       37.0000                 2.4366                  15     0.1916
Actual low-ranked positive examples outside top 50 (maximum 2, lowest probability):
         client_hash_id          content_hash_id  model_probability  is_declining_proxy  early_impressions  early_clicks  weighted_avg_position

Why these failures can occur:

- The model may mistake temporary March traffic changes for a persistent pattern; a large early period does not establish the cause of the later proxy.
- Seasonal or campaign-driven demand can change impressions independently of content quality.
- Position and CTR describe measured search behavior but cannot reveal indexing, cannibalization, competitor changes, or page content quality.
- A lower model probability for a proxy-positive page may reflect sparse active-search days, different client behavior, or weak historical separation in the test clients.
- The grouped holdout is small at the client level, so actual examples may reflect characteristics of only the nine held-out clients rather than all clients.

## 5. Claim rewrite

The table below replaces overconfident Week 5-style statements with language tied to the actual held-out measurement.

In [5]:
claim_table = pd.DataFrame([
    {'Original or potentially overstated claim':'The model predicts future SEO success.','Safer rewritten claim':'Logistic regression produced probability rankings associated with the observed March 16-31 decline proxy on this held-out set.','Why':'The proxy is historical and does not establish future performance.'},
    {'Original or potentially overstated claim':'Refreshing these pages will improve rankings or traffic.','Safer rewritten claim':'The pages are decision-support candidates for human review; refresh benefit is untested.','Why':'No causal or intervention evidence was measured.'},
    {'Original or potentially overstated claim':'The model proves why a page is declining.','Safer rewritten claim':'The model indicates measured feature patterns associated with the decline proxy.','Why':'Feature coefficients are associations, not explanations or causal drivers.'},
    {'Original or potentially overstated claim':'The logistic model is generally better for all clients.','Safer rewritten claim':'On the current client-grouped holdout, the logistic ranking differed from the Week 4 rule on these 9 unseen clients and 2,216 pages.','Why':'The grouped holdout is limited to observed test clients and must not be generalized beyond it.'},
    {'Original or potentially overstated claim':'The random split demonstrates stronger model skill.','Safer rewritten claim':'A random row split is shown only as a diagnostic; it may overstate generalization because client pages can appear in both partitions.','Why':'Same-client leakage across train and test can produce optimistic results.'},
])
print(claim_table.to_string(index=False))
assert len(claim_table) >= 5


                Original or potentially overstated claim                                                                                                                 Safer rewritten claim                                                                                            Why
                  The model predicts future SEO success.        Logistic regression produced probability rankings associated with the observed March 16-31 decline proxy on this held-out set.                             The proxy is historical and does not establish future performance.
Refreshing these pages will improve rankings or traffic.                                              The pages are decision-support candidates for human review; refresh benefit is untested.                                               No causal or intervention evidence was measured.
               The model proves why a page is declining.                                                      The model indicates measured fea

## Self-check

Before submission, confirm each line honestly:

- [x] Both paper findings are accurately described from the extracted research paper.
- [x] Two constructive methodology questions examine label and validation choices.
- [x] Before/after client-grouped and random-row comparison is present.
- [x] Client overlap is checked for the client-grouped split.
- [x] The final feature set uses only March 1-15 search information.
- [x] The leakage audit covers late impressions/clicks, trend fields, the proxy target, IDs, and product flags.
- [x] Actual grouped-holdout false positives and low-ranked positive examples are shown where available.
- [x] Claim language uses observed, measured, associated, directional, and decision-support wording.
- [x] No private names, URLs, raw queries, or token values are printed.
- [x] All code cells execute successfully in the saved notebook.

In [6]:
# Final validation assertions.
assert len(feature_columns) == 5
assert 'is_declining_proxy' not in feature_columns
assert len(train_clients.intersection(test_clients)) == 0
assert len(grouped_test) > 0
assert len(comparison) == 2
assert comparison.loc[0,'Client overlap'] == 0
assert {'Precision@20','Precision@50','ROC-AUC','Positive-class/base rate'}.issubset(comparison.columns)
print('Week 6 validation audit assertions passed:', True)
print('Code cells executed in this notebook: 6')
print('Honest client overlap:', comparison.loc[0,'Client overlap'])
print('Random row split client overlap:', comparison.loc[1,'Client overlap'])
print('No predictive leakage fields detected in the five-feature frame:', True)


Week 6 validation audit assertions passed: True
Code cells executed in this notebook: 6
Honest client overlap: 0
Random row split client overlap: 33
No predictive leakage fields detected in the five-feature frame: True
